# nano-prover: train on Colab (free T4), verify locally

Run **top to bottom in one sitting** (free Colab disconnects when idle — if it drops, the trained
adapter on `/content` is lost, so don't pause between training and the push).

- The kernel runs on a **remote Google GPU machine**; cell 2 clones your repo there.
- The **only thing you must edit** is the `TOKEN` in the last code cell (a GitHub token).
- Default model is the **0.5B** (fast, ~10 min). Switch to 1.5B by editing cell 4 if you have quota.

In [ ]:
# 1. confirm a GPU (must show Tesla T4 / similar). If 'command not found' -> CPU runtime, redo as GPU.
#    If a 'quota' popup appears but this shows a T4, just close the popup — you have a GPU.
!nvidia-smi

In [ ]:
# 2. clone the repo (force-latest) + install the stack.  Keeps untracked files; no nesting.
import os
os.chdir('/content')
if not os.path.isdir('/content/Conjecture-Gym'):
    !git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/content/Conjecture-Gym')
!git fetch -q --depth 1 origin main && git reset -q --hard origin/main
assert os.path.exists('output/train_corpus.jsonl'), 'corpus not found — did you push it?'
!git log --oneline -1
!pip install -q -U transformers trl peft datasets accelerate

In [ ]:
# 3. build train/test splits from the corpus
!python training/prepare_data.py

In [ ]:
# 4. LoRA SFT — 0.5B, ~10 min on T4 (fast + quota-friendly).  -> training/out/ (adapter)
#    Bigger/better: drop SFT_MODEL to use the 1.5B default (~30 min).  OOM? add SFT_GRAD_CKPT=1.
!SFT_MODEL=Qwen/Qwen2.5-Coder-0.5B-Instruct python training/sft_train.py

In [ ]:
# 5. generate proofs for 100 held-out statements (base auto-detected from the adapter).
!python training/gen_preds.py --n 100 --k 1

In [ ]:
# 6. push the small preds.jsonl back to GitHub.  EDIT THE TOKEN below (keep the quotes).
#    Make one: github.com -> Settings -> Developer settings -> Personal access tokens
#              -> Tokens (classic) -> Generate new token -> scope: repo -> copy the ghp_... string.
TOKEN = 'ghp_PUT_YOUR_REAL_TOKEN_HERE'
!git add -f training/preds.jsonl
!git -c user.email=you@example.com -c user.name=you commit -m 'eval preds'
!git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Done on Colab. Now on your **local** machine (where the Lean verifier lives):
```bash
git pull
python training/verify_preds.py     # real Lean-verified pass@k
```